# Stanford RNA 3D Folding Part 2 - Competitive Solution

**Approach:** RNAPro / Protenix + Template-Based Modeling + Multi-Seed Ensemble + Diverse Selection

**Kaggle Notebook Settings:**
- Accelerator: **GPU T4 x2**
- Internet: **OFF**
- Persistence: **Files only**

**Required Kaggle Dataset Inputs (attach before running):**
1. `rnapro-weights` - RNAPro model weights from HuggingFace (nvidia/RNAPro-Public-Best-500M)
2. `ribonanzanet2` - RibonanzaNet2 checkpoint from Kaggle Models
3. `protenix-base` - Protenix base checkpoint (protenix_base_default_v0.5.0.pt)
4. `rna-folding-templates` - Precomputed template CSVs from public template notebooks
5. `rna-deps` - Offline Python wheel dependencies
6. `stanford-rna-3d-folding-2` - Competition data (auto-attached)

In [ ]:
# ============================================================================
# CELL 1: ENVIRONMENT SETUP & CONFIGURATION
# ============================================================================

import os
import sys
import time
import gc
import warnings
import logging

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

# ---- Detect environment ----
IS_KAGGLE = os.path.exists('/kaggle/input')
BASE_INPUT = '/kaggle/input' if IS_KAGGLE else './input'
BASE_OUTPUT = '/kaggle/working' if IS_KAGGLE else './output'
os.makedirs(BASE_OUTPUT, exist_ok=True)

# ---- Dataset paths ----
RNAPRO_WEIGHTS_DIR = os.path.join(BASE_INPUT, 'rnapro-weights')
RIBONANZANET2_DIR = os.path.join(BASE_INPUT, 'ribonanzanet2')
PROTENIX_BASE_DIR = os.path.join(BASE_INPUT, 'protenix-base')
TEMPLATES_DIR = os.path.join(BASE_INPUT, 'rna-folding-templates')
DEPS_DIR = os.path.join(BASE_INPUT, 'rna-deps')
COMPETITION_DATA_DIR = os.path.join(BASE_INPUT, 'stanford-rna-3d-folding-2')

# ---- Install offline dependencies ----
def install_offline_deps():
    if os.path.exists(DEPS_DIR):
        wheels = [f for f in os.listdir(DEPS_DIR) if f.endswith(('.whl', '.tar.gz'))]
        if wheels:
            logger.info(f'Installing {len(wheels)} offline packages...')
            os.system(f'pip install --no-index --find-links={DEPS_DIR} {DEPS_DIR}/*.whl 2>/dev/null')
    
    protenix_src = os.path.join(DEPS_DIR, 'protenix')
    if os.path.exists(protenix_src):
        os.system(f'pip install -e {protenix_src} 2>/dev/null')
    
    rnapro_src = os.path.join(DEPS_DIR, 'RNAPro')
    if os.path.exists(rnapro_src):
        os.system(f'pip install -e {rnapro_src} 2>/dev/null')

install_offline_deps()

# ---- GPU Configuration ----
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field

DEVICE_0 = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
DEVICE_1 = torch.device('cuda:1' if torch.cuda.device_count() > 1 else DEVICE_0)
DTYPE = torch.float16

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}, GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} '
          f'({torch.cuda.get_device_properties(i).total_mem / 1e9:.1f} GB)')

In [ ]:
# ============================================================================
# CELL 2: DATA STRUCTURES & UTILITIES
# ============================================================================

@dataclass
class RNATarget:
    target_id: str
    sequence: str
    length: int = 0

    def __post_init__(self):
        self.length = len(self.sequence)

    @property
    def residue_names(self) -> List[str]:
        return list(self.sequence)

    @property
    def residue_ids(self) -> List[int]:
        return list(range(1, self.length + 1))


@dataclass
class Prediction:
    target_id: str
    coords: np.ndarray  # (n_residues, 3) C1' atom x,y,z
    confidence: float = 0.0
    seed: int = 0
    template_idx: int = -1


@dataclass
class TargetPredictions:
    target: RNATarget
    predictions: List[Prediction] = field(default_factory=list)
    selected_indices: List[int] = field(default_factory=list)


def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def load_test_sequences() -> pd.DataFrame:
    test_path = os.path.join(COMPETITION_DATA_DIR, 'test_sequences.csv')
    if not os.path.exists(test_path):
        for candidate in [
            '/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv',
            './test_sequences.csv',
        ]:
            if os.path.exists(candidate):
                test_path = candidate
                break
    logger.info(f'Loading: {test_path}')
    df = pd.read_csv(test_path)
    logger.info(f'{len(df)} sequences, cols={list(df.columns)}')
    logger.info(f'Lengths: min={df["sequence"].str.len().min()}, '
                f'max={df["sequence"].str.len().max()}, '
                f'mean={df["sequence"].str.len().mean():.0f}')
    return df

print('Data structures loaded.')

In [ ]:
# ============================================================================
# CELL 3: TEMPLATE PROCESSING
# ============================================================================

class TemplateManager:
    def __init__(self, templates_dir: str):
        self.templates_dir = templates_dir
        self.templates: Dict[str, List[np.ndarray]] = {}
        self._load_templates()

    def _load_templates(self):
        if not os.path.exists(self.templates_dir):
            logger.warning(f'Templates dir not found: {self.templates_dir}')
            return

        csv_files = list(Path(self.templates_dir).glob('*.csv'))
        pt_files = list(Path(self.templates_dir).glob('*.pt'))

        if pt_files:
            self._load_pt_templates(pt_files)
        elif csv_files:
            self._load_csv_templates(csv_files)
        logger.info(f'Templates loaded for {len(self.templates)} targets')

    def _load_pt_templates(self, pt_files):
        for pt_file in pt_files:
            try:
                data = torch.load(pt_file, map_location='cpu', weights_only=False)
                if isinstance(data, dict):
                    for tid, coords in data.items():
                        if isinstance(coords, torch.Tensor):
                            coords = coords.numpy()
                        self.templates.setdefault(tid, []).append(coords)
                elif isinstance(data, torch.Tensor):
                    self.templates[pt_file.stem] = [data.numpy()]
            except Exception as e:
                logger.warning(f'Failed to load {pt_file}: {e}')

    def _load_csv_templates(self, csv_files):
        for csv_file in csv_files:
            try:
                df = pd.read_csv(csv_file)
                if 'ID' in df.columns:
                    df['target_id'] = df['ID'].apply(lambda x: '_'.join(str(x).split('_')[:-1]))
                    for tid, group in df.groupby('target_id'):
                        group = group.sort_values('resid' if 'resid' in group.columns else 'ID')
                        templates = []
                        for pidx in range(1, 6):
                            cols = [f'x_{pidx}', f'y_{pidx}', f'z_{pidx}']
                            if all(c in group.columns for c in cols):
                                coords = group[cols].values.astype(np.float32)
                                if not np.all(np.isnan(coords)):
                                    templates.append(coords)
                        if templates:
                            self.templates[str(tid)] = templates
            except Exception as e:
                logger.warning(f'Failed to load CSV {csv_file}: {e}')

    def get_templates(self, target_id: str, max_templates: int = 5) -> List[np.ndarray]:
        return self.templates.get(target_id, [])[:max_templates]

    def has_templates(self, target_id: str) -> bool:
        return target_id in self.templates and len(self.templates[target_id]) > 0

print('TemplateManager loaded.')

In [ ]:
# ============================================================================
# CELL 4: STRUCTURE PREDICTION ENGINE
# ============================================================================

class RNAStructurePredictor:
    """
    RNA 3D structure prediction. Tries models in priority order:
    1. RNAPro (SOTA)
    2. Protenix (strong baseline)
    3. Physics-based fallback
    """

    def __init__(self, template_manager: TemplateManager):
        self.template_manager = template_manager
        self.model = None
        self.model_type = None
        self.ribonanzanet = None
        self._load_best_available_model()

    def _load_best_available_model(self):
        if self._try_load_rnapro():
            self.model_type = 'rnapro'
            logger.info('Loaded RNAPro model (SOTA)')
            return
        if self._try_load_protenix():
            self.model_type = 'protenix'
            logger.info('Loaded Protenix model')
            return
        self.model_type = 'physics'
        logger.warning('No DL model available. Using physics-based fallback.')

    def _try_load_rnapro(self) -> bool:
        try:
            rnapro_ckpt = None
            for pattern in ['*.pt', '*.ckpt', '*.pth', '*.bin']:
                files = list(Path(RNAPRO_WEIGHTS_DIR).glob(pattern))
                if files:
                    rnapro_ckpt = str(files[0])
                    break
            if rnapro_ckpt is None:
                return False

            try:
                from protenix.model.protenix import Protenix
                from protenix.config import parse_configs
            except ImportError:
                sys.path.insert(0, os.path.join(DEPS_DIR, 'RNAPro'))
                sys.path.insert(0, os.path.join(DEPS_DIR, 'protenix'))
                from protenix.model.protenix import Protenix
                from protenix.config import parse_configs

            logger.info(f'Loading RNAPro from: {rnapro_ckpt}')
            self._try_load_ribonanzanet()

            config = parse_configs(
                model_name='rnapro_base',
                load_checkpoint_path=rnapro_ckpt,
                dtype='fp16',
            )
            self.model = Protenix(config)
            self.model.eval()
            self.model = self.model.to(DEVICE_0)
            return True
        except Exception as e:
            logger.warning(f'Failed to load RNAPro: {e}')
            return False

    def _try_load_protenix(self) -> bool:
        try:
            protenix_ckpt = None
            for search_dir in [PROTENIX_BASE_DIR, RNAPRO_WEIGHTS_DIR]:
                if os.path.exists(search_dir):
                    for pattern in ['*.pt', '*.ckpt', '*.pth']:
                        files = list(Path(search_dir).glob(pattern))
                        if files:
                            protenix_ckpt = str(files[0])
                            break
                if protenix_ckpt:
                    break
            if protenix_ckpt is None:
                return False

            try:
                from protenix.model.protenix import Protenix
                from protenix.config import parse_configs
            except ImportError:
                sys.path.insert(0, os.path.join(DEPS_DIR, 'protenix'))
                from protenix.model.protenix import Protenix
                from protenix.config import parse_configs

            logger.info(f'Loading Protenix from: {protenix_ckpt}')
            config = parse_configs(
                model_name='protenix_base',
                load_checkpoint_path=protenix_ckpt,
                dtype='fp16',
            )
            self.model = Protenix(config)
            self.model.eval()
            self.model = self.model.to(DEVICE_0)
            return True
        except Exception as e:
            logger.warning(f'Failed to load Protenix: {e}')
            return False

    def _try_load_ribonanzanet(self):
        try:
            if not os.path.exists(RIBONANZANET2_DIR):
                return
            ckpt_files = list(Path(RIBONANZANET2_DIR).glob('**/*.pt')) + \
                         list(Path(RIBONANZANET2_DIR).glob('**/*.ckpt'))
            if ckpt_files:
                logger.info(f'RibonanzaNet2 found: {ckpt_files[0]}')
                self.ribonanzanet = str(ckpt_files[0])
        except Exception as e:
            logger.warning(f'RibonanzaNet2 load failed: {e}')

    def predict(self, target: RNATarget, n_predictions: int = 10,
                seeds: Optional[List[int]] = None) -> List[Prediction]:
        if seeds is None:
            seeds = list(range(42, 42 + n_predictions))

        predictions = []
        templates = self.template_manager.get_templates(target.target_id)
        has_templates = len(templates) > 0
        n_steps, n_cycle = self._get_adaptive_params(target.length)

        logger.info(f'Predicting {target.target_id} (len={target.length}, '
                     f'templates={len(templates)}, seeds={len(seeds)})')

        for seed_idx, seed in enumerate(seeds):
            try:
                template_idx = seed_idx % max(len(templates), 1) if has_templates else -1
                template_subset = [templates[template_idx]] if has_templates and template_idx < len(templates) else None

                if self.model_type in ('rnapro', 'protenix'):
                    pred = self._predict_with_model(target, seed, template_subset, n_steps, n_cycle)
                else:
                    pred = self._predict_physics_fallback(target, seed, template_subset)

                if pred is not None:
                    pred.seed = seed
                    pred.template_idx = template_idx
                    predictions.append(pred)
            except torch.cuda.OutOfMemoryError:
                logger.warning(f'OOM seed={seed}, trying reduced params...')
                clear_gpu_memory()
                try:
                    pred = self._predict_with_reduced_params(target, seed, templates)
                    if pred is not None:
                        predictions.append(pred)
                except Exception:
                    clear_gpu_memory()
            except Exception as e:
                logger.error(f'Failed seed={seed}: {e}')
                clear_gpu_memory()

        if not predictions and has_templates:
            predictions = self._template_only_predictions(target, templates)
        if not predictions:
            for seed in seeds[:5]:
                pred = self._predict_physics_fallback(target, seed, None)
                if pred is not None:
                    predictions.append(pred)

        logger.info(f'Generated {len(predictions)} predictions')
        return predictions

    def _get_adaptive_params(self, seq_length: int) -> Tuple[int, int]:
        if seq_length > 4000: return 50, 1
        elif seq_length > 2000: return 100, 2
        elif seq_length > 1000: return 150, 3
        else: return 200, 4

    def _predict_with_model(self, target, seed, templates, n_steps, n_cycle):
        torch.manual_seed(seed)
        np.random.seed(seed)
        with torch.no_grad(), torch.cuda.amp.autocast(dtype=DTYPE):
            input_data = self._prepare_model_input(target, templates)
            output = self.model.inference(input_data, n_step=n_steps, n_cycle=n_cycle, seed=seed)
            coords = self._extract_c1_coords(output, target.length)
            confidence = self._extract_confidence(output)
            return Prediction(target_id=target.target_id, coords=coords, confidence=confidence, seed=seed)

    def _predict_with_reduced_params(self, target, seed, templates):
        clear_gpu_memory()
        tmpl = [templates[0]] if templates else None
        return self._predict_with_model(target, seed, tmpl, n_steps=50, n_cycle=1)

    def _prepare_model_input(self, target, templates):
        nuc_map = {'A': 0, 'U': 1, 'G': 2, 'C': 3, 'N': 4}
        seq_encoded = [nuc_map.get(n, 4) for n in target.sequence]
        input_data = {
            'sequence': target.sequence,
            'seq_encoded': torch.tensor(seq_encoded, dtype=torch.long).unsqueeze(0),
            'target_id': target.target_id,
            'seq_length': target.length,
        }
        if templates:
            input_data['templates'] = torch.stack([torch.tensor(t, dtype=torch.float32) for t in templates]).unsqueeze(0)
            input_data['num_templates'] = len(templates)
        if self.ribonanzanet is not None:
            input_data['ribonanzanet_path'] = self.ribonanzanet
        return input_data

    def _extract_c1_coords(self, output, n_residues):
        for key in ['final_atom_positions', 'atom_positions', 'coords', 'positions', 'pred_coords']:
            if key in output:
                c = output[key]
                if isinstance(c, torch.Tensor): c = c.cpu().numpy()
                if c.ndim == 4: c = c[0]
                if c.ndim == 3: return c[:n_residues, 0, :].astype(np.float32)
                return c[:n_residues, :].astype(np.float32)
        raise ValueError(f'Cannot extract coords. Keys: {list(output.keys())}')

    def _extract_confidence(self, output):
        for key in ['plddt', 'confidence', 'iptm', 'ptm', 'ranking_confidence']:
            if key in output:
                v = output[key]
                if isinstance(v, torch.Tensor): v = v.cpu().item() if v.numel() == 1 else v.cpu().mean().item()
                return float(v)
        return 0.5

    def _template_only_predictions(self, target, templates):
        predictions = []
        for i, tmpl in enumerate(templates[:5]):
            if len(tmpl) != target.length:
                tmpl = self._resize_template(tmpl, target.length)
            np.random.seed(42 + i)
            noise = np.random.randn(*tmpl.shape).astype(np.float32) * 0.5
            predictions.append(Prediction(
                target_id=target.target_id,
                coords=(tmpl + noise).astype(np.float32),
                confidence=0.7 - 0.05 * i,
                seed=42 + i, template_idx=i,
            ))
        return predictions

    def _predict_physics_fallback(self, target, seed, templates):
        np.random.seed(seed)
        n = target.length
        if templates and len(templates) > 0:
            tmpl = templates[0]
            if len(tmpl) == n:
                noise = np.random.randn(n, 3).astype(np.float32) * (1.0 + seed * 0.1)
                return Prediction(target_id=target.target_id, coords=tmpl + noise, confidence=0.4, seed=seed)
        coords = self._generate_coarse_grained_structure(target.sequence, seed)
        return Prediction(target_id=target.target_id, coords=coords, confidence=0.2, seed=seed)

    def _generate_coarse_grained_structure(self, sequence, seed):
        np.random.seed(seed)
        n = len(sequence)
        pairs = self._nussinov_fold(sequence)
        coords = np.zeros((n, 3), dtype=np.float32)
        RISE, TWIST, RADIUS = 2.81, 32.7, 9.4
        paired_set = set()
        for p in pairs:
            paired_set.add(p[0])
            paired_set.add(p[1])
        for i in range(n):
            angle = np.radians(TWIST * i + seed * 37)
            x = RADIUS * np.cos(angle)
            y = RADIUS * np.sin(angle)
            z = RISE * i
            if i not in paired_set:
                x += np.random.randn() * 3.0
                y += np.random.randn() * 3.0
                z += np.random.randn() * 3.0
            coords[i] = [x, y, z]
        coords -= coords.mean(axis=0)
        return self._remove_clashes(coords)

    def _nussinov_fold(self, sequence, min_loop=4):
        n = len(sequence)
        # For very long sequences, skip expensive DP
        if n > 2000:
            return []
        can_pair = {('A','U'):1,('U','A'):1,('G','C'):1,('C','G'):1,('G','U'):1,('U','G'):1}
        dp = np.zeros((n, n), dtype=np.int32)
        for span in range(min_loop + 1, n):
            for i in range(n - span):
                j = i + span
                dp[i][j] = dp[i+1][j]
                dp[i][j] = max(dp[i][j], dp[i][j-1])
                if can_pair.get((sequence[i], sequence[j]), 0):
                    dp[i][j] = max(dp[i][j], (dp[i+1][j-1] if i+1<=j-1 else 0) + 1)
                for k in range(i+1, j):
                    dp[i][j] = max(dp[i][j], dp[i][k] + dp[k+1][j])
        pairs = []
        self._traceback(dp, sequence, 0, n-1, pairs, can_pair, min_loop)
        return pairs

    def _traceback(self, dp, seq, i, j, pairs, can_pair, min_loop):
        if i >= j: return
        if dp[i][j] == dp[i+1][j]:
            self._traceback(dp, seq, i+1, j, pairs, can_pair, min_loop)
        elif dp[i][j] == dp[i][j-1]:
            self._traceback(dp, seq, i, j-1, pairs, can_pair, min_loop)
        elif can_pair.get((seq[i], seq[j]), 0) and dp[i][j] == (dp[i+1][j-1] if i+1<=j-1 else 0) + 1:
            pairs.append((i, j))
            self._traceback(dp, seq, i+1, j-1, pairs, can_pair, min_loop)
        else:
            for k in range(i+1, j):
                if dp[i][j] == dp[i][k] + dp[k+1][j]:
                    self._traceback(dp, seq, i, k, pairs, can_pair, min_loop)
                    self._traceback(dp, seq, k+1, j, pairs, can_pair, min_loop)
                    break

    def _remove_clashes(self, coords, min_dist=3.0, n_iter=100):
        n = len(coords)
        for _ in range(n_iter):
            moved = False
            for i in range(n):
                for j in range(i+2, min(i+10, n)):
                    diff = coords[j] - coords[i]
                    dist = np.linalg.norm(diff)
                    if dist < min_dist and dist > 0.01:
                        push = (min_dist - dist) / 2.0 * diff / dist
                        coords[i] -= push
                        coords[j] += push
                        moved = True
            if not moved: break
        return coords

    def _resize_template(self, template, target_length):
        from scipy.interpolate import interp1d
        old = np.linspace(0, 1, len(template))
        new = np.linspace(0, 1, target_length)
        result = np.zeros((target_length, 3), dtype=np.float32)
        for d in range(3):
            result[:, d] = interp1d(old, template[:, d], kind='linear')(new)
        return result

print('RNAStructurePredictor loaded.')

In [ ]:
# ============================================================================
# CELL 5: DIVERSE PREDICTION SELECTION
# ============================================================================

class PredictionSelector:
    """
    Select best 5 predictions from N candidates.
    Balances confidence + structural diversity (best-of-5 scoring).
    """

    @staticmethod
    def compute_rmsd(coords1, coords2):
        c1 = coords1 - coords1.mean(axis=0)
        c2 = coords2 - coords2.mean(axis=0)
        min_len = min(len(c1), len(c2))
        c1, c2 = c1[:min_len], c2[:min_len]
        H = c1.T @ c2
        U, S, Vt = np.linalg.svd(H)
        d = np.linalg.det(Vt.T @ U.T)
        sign_matrix = np.eye(3)
        sign_matrix[2, 2] = np.sign(d)
        R = Vt.T @ sign_matrix @ U.T
        c2_rotated = (R @ c2.T).T
        return np.sqrt(np.mean(np.sum((c1 - c2_rotated) ** 2, axis=1)))

    @staticmethod
    def select_diverse_top5(predictions, confidence_weight=0.6, diversity_weight=0.4):
        if len(predictions) <= 5:
            return list(range(len(predictions)))

        n = len(predictions)
        confidences = np.array([p.confidence for p in predictions])
        r = confidences.max() - confidences.min()
        norm_conf = (confidences - confidences.min()) / r if r > 0 else np.ones(n)

        # Pairwise RMSD
        rmsd_matrix = np.zeros((n, n), dtype=np.float32)
        for i in range(n):
            for j in range(i + 1, n):
                r_val = PredictionSelector.compute_rmsd(predictions[i].coords, predictions[j].coords)
                rmsd_matrix[i, j] = rmsd_matrix[j, i] = r_val

        max_rmsd = rmsd_matrix.max()
        norm_rmsd = rmsd_matrix / max_rmsd if max_rmsd > 0 else rmsd_matrix

        selected = [int(np.argmax(norm_conf))]  # Start with highest confidence
        available = set(range(n)) - set(selected)

        for _ in range(4):
            if not available: break
            best_score, best_idx = -float('inf'), -1
            for idx in available:
                conf_score = norm_conf[idx] * confidence_weight
                div_score = min(norm_rmsd[idx, s] for s in selected) * diversity_weight
                total = conf_score + div_score
                if total > best_score:
                    best_score, best_idx = total, idx
            if best_idx >= 0:
                selected.append(best_idx)
                available.discard(best_idx)

        return selected

print('PredictionSelector loaded.')

In [ ]:
# ============================================================================
# CELL 6: SUBMISSION GENERATION
# ============================================================================

def generate_submission(targets, all_predictions, output_path):
    rows = []
    for target in targets:
        tp = all_predictions.get(target.target_id)
        if tp is None or len(tp.predictions) == 0:
            logger.warning(f'No predictions for {target.target_id}')
            for ri in range(target.length):
                row = {'ID': f'{target.target_id}_{ri+1}', 'resname': target.sequence[ri], 'resid': ri+1}
                for pi in range(1, 6):
                    row[f'x_{pi}'] = row[f'y_{pi}'] = row[f'z_{pi}'] = 0.0
                rows.append(row)
            continue

        selected = tp.selected_indices
        preds = [tp.predictions[i] for i in selected]

        # Pad to 5
        while len(preds) < 5:
            if preds:
                last = preds[-1]
                noise = np.random.randn(*last.coords.shape).astype(np.float32) * 0.3
                preds.append(Prediction(target_id=target.target_id, coords=last.coords + noise, confidence=last.confidence * 0.9))
            else:
                preds.append(Prediction(target_id=target.target_id, coords=np.zeros((target.length, 3), dtype=np.float32)))

        for ri in range(target.length):
            row = {'ID': f'{target.target_id}_{ri+1}', 'resname': target.sequence[ri], 'resid': ri+1}
            for pi in range(5):
                c = preds[pi].coords
                if ri < len(c):
                    row[f'x_{pi+1}'] = float(c[ri, 0])
                    row[f'y_{pi+1}'] = float(c[ri, 1])
                    row[f'z_{pi+1}'] = float(c[ri, 2])
                else:
                    row[f'x_{pi+1}'] = row[f'y_{pi+1}'] = row[f'z_{pi+1}'] = 0.0
            rows.append(row)

    columns = ['ID', 'resname', 'resid']
    for i in range(1, 6):
        columns.extend([f'x_{i}', f'y_{i}', f'z_{i}'])
    df = pd.DataFrame(rows, columns=columns)
    df.to_csv(output_path, index=False)
    logger.info(f'Submission: {df.shape}, saved to {output_path}')
    return df

print('Submission generator loaded.')

In [ ]:
# ============================================================================
# CELL 7: MAIN PIPELINE - RUN EVERYTHING
# ============================================================================

start_time = time.time()

# ---- Load test data ----
print('=' * 60)
print('STEP 1: Loading test sequences')
print('=' * 60)
test_df = load_test_sequences()
targets = []
for _, row in test_df.iterrows():
    tid = row.get('target_id', row.get('ID', row.get('id', '')))
    seq = row.get('sequence', row.get('seq', ''))
    targets.append(RNATarget(target_id=str(tid), sequence=str(seq)))
targets.sort(key=lambda t: t.length)
print(f'{len(targets)} targets, lengths: {[t.length for t in targets[:5]]}...{[t.length for t in targets[-3:]]}')

# ---- Load templates ----
print('=' * 60)
print('STEP 2: Loading templates')
print('=' * 60)
template_manager = TemplateManager(TEMPLATES_DIR)
print(f'Targets with templates: {sum(1 for t in targets if template_manager.has_templates(t.target_id))}/{len(targets)}')

# ---- Initialize predictor ----
print('=' * 60)
print('STEP 3: Initializing predictor')
print('=' * 60)
predictor = RNAStructurePredictor(template_manager)
print(f'Model type: {predictor.model_type}')

# ---- Generate predictions ----
print('=' * 60)
print('STEP 4: Generating predictions')
print('=' * 60)

all_predictions = {}
selector = PredictionSelector()

time_budget = 8.5 * 3600
preds_per_target = max(5, min(20, int(time_budget / 60) // max(len(targets), 1)))
base_seeds = [42, 137, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768,
              7, 13, 23, 37, 53, 71, 97, 113, 151, 199]

for tidx, target in enumerate(targets):
    elapsed = time.time() - start_time
    remaining = time_budget - elapsed
    if remaining < 300 and tidx > 0:
        preds_per_target = 5

    print(f'\n--- Target {tidx+1}/{len(targets)}: {target.target_id} (len={target.length}) ---')

    n_seeds = min(preds_per_target, len(base_seeds))
    seeds = base_seeds[:n_seeds]

    predictions = predictor.predict(target, n_predictions=n_seeds, seeds=seeds)
    if not predictions:
        print(f'WARNING: No predictions for {target.target_id}')
        continue

    selected = selector.select_diverse_top5(predictions) if len(predictions) > 5 else list(range(len(predictions)))

    all_predictions[target.target_id] = TargetPredictions(
        target=target, predictions=predictions, selected_indices=selected,
    )
    print(f'Selected {len(selected)} preds, confidences: {[f"{predictions[i].confidence:.3f}" for i in selected]}')
    clear_gpu_memory()

# ---- Generate submission ----
print('=' * 60)
print('STEP 5: Generating submission.csv')
print('=' * 60)
submission_path = os.path.join(BASE_OUTPUT, 'submission.csv')
submission_df = generate_submission(targets, all_predictions, submission_path)

total_time = time.time() - start_time
print('=' * 60)
print(f'DONE! Time: {total_time/60:.1f} min, Shape: {submission_df.shape}')
print(f'Output: {submission_path}')
print('=' * 60)
submission_df.head(10)